# Replicating Tessler et al. (2024) Fig. 4D — HM "majority bias" vs group movement toward the majority

Main text (RQ3): *"discussants might have gravitated toward the majority view simply because they were asked to judge several
group statements that supported that position. To test this, we measured the relationship between participants' viewpoint change
toward the majority position (from pre- to postdeliberation position ratings) and the fraction of group statements whose position
component scores fell on the majority side of the median opinion ('majority bias'). We found no relationship between fractional
exposure to majority views and subsequent change in viewpoint toward the majority (b = 0.058, SE = 0.07, z score = 0.9, P = 0.37)."*
Caption: *"Individual points represent a single group discussing a single question."* Panel D's x values sit at multiples of 1/8
(4 initial + 4 revised candidates) and its y values at multiples of 1/5 and 1/4 (group sizes), which fixes the y variable as a
group mean of a per-participant {-1, 0, +1} quantity.

Definitions used here (`hm_fig4c/fig4d.py`): majority direction from the pre-deliberation ratings (tie → AGREE, SM 4.1.2.1);
majority-aligned rating x' = 8 − x when the majority is DISAGREE; majority bias = share of the round's 8 candidates whose
position score lies on the majority side of the median opinion score; movement = mean of sign(post' − pre') over the group.
The paper does not state its random-effects structure for this test, so four estimators are reported.

In [ ]:
import os, sys, json
sys.path.insert(0, os.path.abspath(".."))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from hm_fig4c import pipeline as P
from hm_fig4c import fig4d as F
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

EMB_DIR = os.environ.get("HM_EMB_DIR", "../embeddings/st5-large")
AXIS_METHOD = os.environ.get("HM_AXIS_METHOD", "unit")
ENDPOINTS = os.environ.get("HM_ENDPOINTS", "prefixed")
MODEL_TAG = os.path.basename(EMB_DIR.rstrip("/"))
OUT_DIR = f"../results/{MODEL_TAG}"; os.makedirs(OUT_DIR, exist_ok=True)
print(EMB_DIR, AXIS_METHOD, ENDPOINTS)

## 1. Position scores and the per-round table

In [ ]:
opinions, statements, questions, candidates = P.score_all("../prepared", EMB_DIR, method=AXIS_METHOD, endpoint_style=ENDPOINTS)
rounds = F.round_table(opinions, candidates, cohort="cohorts_1_3", prereg_only=True, majority_by="rating")
rounds.to_csv(f"{OUT_DIR}/fig4d_rounds.csv", index=False)
print("rounds:", len(rounds), "| groups:", rounds["launch_id"].nunique(), "| questions:", rounds["question_id"].nunique())
print("candidates per round:", rounds["n_candidates"].value_counts().to_dict(), "| group size:", rounds["n"].value_counts().to_dict())
print("participants with both ratings per round:", rounds["n_both"].value_counts().sort_index().to_dict())
print("ties (majority set to AGREE):", int(rounds["tie"].sum()), "| rounds with a minority:", int(rounds["has_minority"].sum()))
rounds.head()

## 2. Raw distributions before any model

In [ ]:
COLS = ["majority_bias", "bias_initial", "bias_revised", "movement_sign", "movement_mean", "movement_gai", "gai_pre"]
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
rounds["majority_bias"].value_counts().sort_index().plot.bar(ax=axes[0], color="grey"); axes[0].set_title("HM majority bias (share of 8 candidates)", fontsize=9)
axes[0].set_xticklabels([f"{v:.3f}" for v in sorted(rounds["majority_bias"].unique())], rotation=90, fontsize=7)
rounds["movement_sign"].round(3).value_counts().sort_index().plot.bar(ax=axes[1], color="grey"); axes[1].set_title("Group movement to majority (mean sign)", fontsize=9)
axes[1].set_xticklabels([f"{v:.2f}" for v in sorted(rounds["movement_sign"].round(3).unique())], rotation=90, fontsize=7)
axes[2].bar(["initial", "revised", "all"], [rounds["bias_initial"].mean(), rounds["bias_revised"].mean(), rounds["majority_bias"].mean()], color=["#c6dbef", "#807dba", "grey"])
axes[2].axhline(.5, ls=":", color="k"); axes[2].set_title("Mean share of candidates on the majority side", fontsize=9)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig4d_distributions.png", dpi=150)
summary_dist = pd.DataFrame({"mean": rounds[COLS].mean(), "sd": rounds[COLS].std(), "share_positive": (rounds[COLS] > 0).mean(), "n": rounds[COLS].notna().sum()})
summary_dist.round(3)

## 3. Fig. 4D — association between majority bias and movement toward the majority (paper: b = 0.058, SE = 0.07, z = 0.9, P = 0.37)

In [ ]:
fit = F.fit_association(rounds, "majority_bias", "movement_sign")
fit.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(3.8, 3.6))
F.plot_fig4d(rounds, fit, ax=ax)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig4d.png", dpi=200)

In [ ]:
# mean movement within each majority-bias band (what the regression line averages over)
band = rounds.groupby("majority_bias")["movement_sign"].agg(["mean", "sem", "size"]); band.round(3)

## 4. Sensitivity
Each row changes one choice: the movement measure (three definitions), the side rule (majority by rating vs by the sign of the
opinion scores), the candidate set (initial or revised only), the sample (rounds with a genuine minority; unanimous rounds; ties
excluded; no pre-registration filter; cohort 4), and adjustment for the group's pre-deliberation agreement (a ceiling on movement).
The slope is from the mixed model with a random intercept by group (falling back to OLS if that fit fails), z = b/SE.

In [ ]:
def slope(df, x="majority_bias", y="movement_sign", covariates=()):
    f = F.fit_association(df, x, y, covariates=covariates, crossed=False)
    r = f.iloc[2] if "b" in f.columns and pd.notna(f.iloc[2].get("b", np.nan)) else f.iloc[0]
    return {"b": r["b"], "se": r["se"], "z": r["z"], "p": r["p"], "n_rounds": int(r["n_rounds"])}
rounds_score = F.round_table(opinions, candidates, cohort="cohorts_1_3", prereg_only=True, majority_by="score")
sens = {
  "primary: mean sign(post' - pre'), 8 candidates, majority by rating": slope(rounds),
  "movement = mean (post' - pre')": slope(rounds, y="movement_mean"),
  "movement = Group Agreement Index change": slope(rounds, y="movement_gai"),
  "majority side by sign of opinion scores": slope(rounds_score),
  "bias from the 4 initial candidates only": slope(rounds, x="bias_initial"),
  "bias from the 4 revised candidates only": slope(rounds, x="bias_revised"),
  "adjusted for pre-deliberation Group Agreement Index": slope(rounds, covariates=("gai_pre",)),
  "rounds with a minority only": slope(rounds[rounds["has_minority"]]),
  "unanimous rounds only": slope(rounds[~rounds["has_minority"]]),
  "tied rounds excluded": slope(rounds[~rounds["tie"]]),
  "all cohort 1-3 rounds (no pre-registration filter)": slope(F.round_table(opinions, candidates, cohort="cohorts_1_3", prereg_only=False)),
  "cohort 4 (critique exclusion)": slope(F.round_table(opinions, candidates, cohort="cohort4")),
}
sens = pd.DataFrame(sens).T; sens.to_csv(f"{OUT_DIR}/fig4d_sensitivity.csv"); sens.round(3)

## 5. Summary vs paper

In [ ]:
primary = fit.iloc[0]; mixed = fit.iloc[2] if pd.notna(fit.iloc[2].get("b", np.nan)) else fit.iloc[0]
ours = {"n_rounds": int(primary["n_rounds"]), "b_ols": primary["b"], "se_ols": primary["se"], "z_ols": primary["z"], "p_ols": primary["p"],
        "b_mixed": mixed["b"], "se_mixed": mixed["se"], "z_mixed": mixed["z"], "p_mixed": mixed["p"],
        "mean_majority_bias": rounds["majority_bias"].mean(), "mean_movement_sign": rounds["movement_sign"].mean(),
        "model": MODEL_TAG, "endpoints": ENDPOINTS, "axis": AXIS_METHOD}
json.dump({"paper": F.PAPER, "ours": ours, "fits": fit.to_dict(orient="records")}, open(f"{OUT_DIR}/fig4d.json", "w"), indent=1, default=float)
pd.DataFrame({"paper": {"b": F.PAPER["b"], "se": F.PAPER["se"], "z": F.PAPER["z"], "p": F.PAPER["p"]},
              "ours (OLS)": {"b": primary["b"], "se": primary["se"], "z": primary["z"], "p": primary["p"]},
              "ours (mixed, group intercept)": {"b": mixed["b"], "se": mixed["se"], "z": mixed["z"], "p": mixed["p"]}}).round(3)